# Stage B2 — does the lens actually surface the intermediate?

Control A ablates the ten directions the lens ranks highest, and expects two-hop reasoning to collapse. That only follows **if those ten directions contain the unspoken intermediate.**

For *"the language spoken in the country where the Amazon River ends"*, the paper's claim is that `Brazil` sits in the readout before the model says `Portuguese`. Nothing has checked whether that happens on your model.

`probe-swap.json` gives the intermediate for all 90 prompts, and **none of them appears anywhere in its own prompt** — so this is a clean test of unspoken content, not an echo of the input.

**Cost:** a few minutes, roughly 1 unit. **Run this before Control A**, not after: if the readout doesn't surface intermediates, that changes how you read the ablation result, and learning it afterwards invites explaining a null with whatever the readout happens to show.

### Two measures

- **rank** — where the intermediate lands in the readout. Low means surfaced.
- **loading** — cosine similarity between the residual stream and the intermediate's lens vector. The paper's *workspace loading*: it predicts whether interventions on a concept work. Number words load poorly and intervene poorly.

### The control that matters

For each prompt the same measures are computed for **another prompt's intermediate**. Without that, a good rank might just mean the lens likes common words. **The true-vs-foil gap is the evidence**, not the raw rate.

## Cell 1 — Setup

In [ ]:
import os, sys, subprocess, torch

!pip -q install -U transformers accelerate huggingface_hub datasets

if not os.path.isdir("/content/jacobian-lens"):
    subprocess.run(["git","clone","-q","--depth","1",
                    "https://github.com/anthropics/jacobian-lens.git"],
                   cwd="/content", check=True)

for p in ("/content/ablation", "/content/band"):
    os.makedirs(p, exist_ok=True); open(f"{p}/__init__.py","a").close()
for p in ("/content/jacobian-lens", "/content"):
    if p not in sys.path: sys.path.insert(0, p)
os.environ["PYTHONPATH"] = "/content/jacobian-lens:/content"

import importlib; importlib.invalidate_caches()
import jlens
print("jlens OK |", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

## Cell 2 — Write `ablation/directions.py`

In [ ]:
%%writefile ablation/directions.py
"""Direction selection and subspace projection for J-space / R-space ablation.

Implements the direction-set half of the ablation harness. Every selector here
produces a set of residual-stream directions of a specified size; the harness
projects them out. Keeping selection separate from projection is what makes the
matched controls of proposal 4.8 cheap: same projection, different selector.

Terminology (guide 2.1): nothing here is "the workspace". These are candidate
directions until Phase 3 says otherwise.
"""

from __future__ import annotations

from dataclasses import dataclass

import torch


# --- J-lens vector construction -------------------------------------------

def lens_vectors(
    unembed_weight: torch.Tensor, jacobian: torch.Tensor, token_ids: torch.Tensor
) -> torch.Tensor:
    """The J-lens vectors for ``token_ids`` at one layer.

    Paper §2.1 defines the J-lens vectors as the rows of ``W_U J_l``. Only the
    requested rows are materialised: the full product is ``[vocab, d_model]``
    and is far too large to hold for a real vocabulary.

    Args:
        unembed_weight: ``W_U``, shape ``[vocab, d_model]``.
        jacobian: ``J_l``, shape ``[d_model, d_model]``.
        token_ids: Shape ``[..., k]``.

    Returns:
        Shape ``[..., k, d_model]``.
    """
    rows = unembed_weight.index_select(0, token_ids.reshape(-1).to(unembed_weight.device))
    rows = rows.to(jacobian.dtype) @ jacobian
    return rows.reshape(*token_ids.shape, jacobian.shape[-1])


# --- Selectors -------------------------------------------------------------

def select_by_rank(
    lens_logits: torch.Tensor,
    k: int,
    *,
    rank_offset: int = 0,
    excluded: torch.Tensor | None = None,
) -> torch.Tensor:
    """Token ids ranked ``rank_offset .. rank_offset + k`` by lens score.

    ``rank_offset=0`` gives the top-k the paper ablates. ``rank_offset=k`` gives
    the next-k, which is the "matched but not selected" control: same lens, same
    size, adjacent rank band. A candidate subspace that matters no more than the
    next-k has not earned H1 (proposal 4.8, extended per the probe-swap design).

    Args:
        lens_logits: Shape ``[n_positions, vocab]``.
        k: Number of directions.
        rank_offset: Rank to start from.
        excluded: Boolean mask ``[n_positions, vocab]``; True entries are never
            selected. This carries the clean-pass exclusion — see
            :func:`clean_top_mask`.

    Returns:
        Shape ``[n_positions, k]``.
    """
    scores = lens_logits.clone()
    if excluded is not None:
        scores = scores.masked_fill(excluded, float("-inf"))
    top = scores.topk(rank_offset + k, dim=-1).indices
    return top[:, rank_offset:]


def random_lens_tokens(
    n_positions: int,
    k: int,
    vocab_size: int,
    generator: torch.Generator,
    *,
    excluded: torch.Tensor | None = None,
    device: torch.device | None = None,
) -> torch.Tensor:
    """Uniformly random token ids — matched-size random control (proposal 4.8).

    Drawn from the lens dictionary rather than isotropically, so the control
    asks "is it *these* lens directions, or any lens directions?". Strictly the
    harder question of the two; run both.
    """
    out = torch.empty(n_positions, k, dtype=torch.long, device=device)
    for p in range(n_positions):
        while True:
            cand = torch.randint(
                vocab_size, (k,), generator=generator, device=generator.device
            ).to(device)
            if excluded is None or not bool(excluded[p, cand].any()):
                out[p] = cand
                break
    return out


def random_isotropic(
    n_positions: int, k: int, d_model: int, generator: torch.Generator,
    *, device: torch.device | None = None, dtype: torch.dtype = torch.float32,
) -> torch.Tensor:
    """Isotropic random unit directions — the paper's random-direction control."""
    v = torch.randn(
        n_positions, k, d_model, generator=generator, device=generator.device,
        dtype=torch.float32,
    ).to(device=device, dtype=dtype)
    return v / v.norm(dim=-1, keepdim=True).clamp_min(1e-12)


def clean_top_mask(
    clean_next_token_logits: torch.Tensor, n_exclude: int = 10
) -> torch.Tensor:
    """Mask marking the clean pass's top-``n_exclude`` predictions per position.

    **This is the confound guard, and it is not optional.** Paper: "we do not
    ablate any tokens that appear in the top-10 tokens of a clean forward pass,
    so as to specifically target the J-space's effects on internal reasoning
    rather than report." Without it, ablation suppresses whatever the model was
    about to say, performance drops for a trivial reason, and H1 gets
    "confirmed" by an artifact.

    Args:
        clean_next_token_logits: Shape ``[n_positions, vocab]`` from an
            unablated forward pass.

    Returns:
        Boolean ``[n_positions, vocab]``, True where a token must not be ablated.
    """
    mask = torch.zeros_like(clean_next_token_logits, dtype=torch.bool)
    top = clean_next_token_logits.topk(n_exclude, dim=-1).indices
    return mask.scatter(-1, top, True)


# --- Projection ------------------------------------------------------------

@dataclass(frozen=True)
class Basis:
    """An orthonormal-row basis with a validity mask for rank-deficient sets."""

    rows: torch.Tensor   # [n_positions, k, d_model], orthonormal rows
    keep: torch.Tensor   # [n_positions, k], float 1/0
    rank: torch.Tensor   # [n_positions], effective rank actually removed


def orthonormalise(vectors: torch.Tensor, *, rtol: float = 1e-6) -> Basis:
    """Orthonormal basis for the span of each position's direction set.

    J-lens vectors are overcomplete and non-orthogonal (paper §2.3), so a set of
    k of them may span fewer than k dimensions. Small singular values are masked
    out rather than dropped, which keeps the operation batched and makes the
    effective rank observable — report it, because "we ablated k directions" is
    false if the span was smaller.
    """
    vectors = vectors.to(torch.float32)
    _, s, vh = torch.linalg.svd(vectors, full_matrices=False)
    keep = (s > rtol * s[..., :1].clamp_min(1e-30)).to(vectors.dtype)
    return Basis(rows=vh, keep=keep, rank=keep.sum(-1))


def project_out(
    hidden: torch.Tensor, basis: Basis, *, mode: str = "subspace"
) -> torch.Tensor:
    """Remove the component of ``hidden`` inside the spanned subspace.

    Args:
        hidden: Shape ``[n_positions, d_model]``.
        mode: ``"subspace"`` projects onto the orthogonal complement of the span
            in one step. ``"sequential"`` removes each direction in turn, which
            is order-dependent for non-orthogonal vectors and therefore removes
            *less* than the full span.

    The paper's phrasing — "zero out the residual stream's projection onto
    each" — does not disambiguate these, and for non-orthogonal J-lens vectors
    they differ. ``"subspace"`` is the default because it is the one that
    actually removes the content; ``"sequential"`` is provided so the choice can
    be tested rather than assumed. Record which was used.
    """
    h = hidden.to(torch.float32)
    if mode == "subspace":
        coeffs = torch.einsum("prd,pd->pr", basis.rows, h) * basis.keep
        return (h - torch.einsum("pr,prd->pd", coeffs, basis.rows)).to(hidden.dtype)
    if mode == "sequential":
        for i in range(basis.rows.shape[1]):
            v = basis.rows[:, i, :] * basis.keep[:, i : i + 1]
            h = h - (h * v).sum(-1, keepdim=True) * v
        return h.to(hidden.dtype)
    raise ValueError(f"unknown mode {mode!r}")

## Cell 3 — Write `ablation/harness.py`

In [ ]:
%%writefile ablation/harness.py
"""Two-pass ablation harness.

Pass 1 is a clean forward pass: it records the residual stream at every band
layer, computes lens logits, and captures the clean next-token distribution.
Pass 2 re-runs with the selected directions projected out.

Two passes are not an optimisation choice — the confound guard of proposal 4.4
(paper: exclude the clean pass's top-10) *requires* knowing the clean output
before choosing what to ablate.

This harness is built to Phase 3 requirements from the first line, per guide
§3a: Control A, Control B, and the Phase 3 sweep all run through it unchanged.
"""

from __future__ import annotations

from dataclasses import dataclass, field, asdict
from typing import Any, Literal, Sequence

import torch

from jlens.hooks import ActivationRecorder
from jlens.lens import JacobianLens

from .directions import (
    Basis,
    clean_top_mask,
    lens_vectors,
    orthonormalise,
    project_out,
    random_isotropic,
    random_lens_tokens,
    select_by_rank,
)

Selector = Literal["topk", "next_k", "random_lens", "random_iso", "none"]


def record_at_or(spec, final):
    return sorted({*spec.layers, final})


def _mask_from_ids(ids: torch.Tensor, shape) -> torch.Tensor:
    """Rebuild the clean-top-k boolean mask from cached ids."""
    m = torch.zeros(shape, dtype=torch.bool, device=ids.device)
    return m.scatter(-1, ids, True)


@dataclass(frozen=True)
class AblationSpec:
    """One fully-specified ablation condition. Serialise this into every result.

    Attributes:
        layers: Band of block indices to ablate at. Light/medium/heavy differ
            only here — the paper varies the layer range, not k.
        k: Directions removed per position (proposal 4.7 sweeps this; the paper
            fixed it at 10). Sweeping k *and* layers multiplies runs — state
            which axis in prereg_phase3.md.
        selector: Which directions. ``"none"`` is the clean baseline.
        seed: Required for every random selector (guide §1.2).
        exclude_clean_top: Confound guard size. **Do not set to 0** except as a
            deliberate, logged demonstration of the artifact it prevents.
        mode: Projection mode; see :func:`project_out`.
        positions: Token positions to ablate at; ``None`` means all.
    """

    layers: tuple[int, ...]
    k: int
    selector: Selector = "topk"
    seed: int | None = None
    exclude_clean_top: int = 10
    mode: str = "subspace"
    positions: tuple[int, ...] | None = None

    def __post_init__(self) -> None:
        if self.selector in ("random_lens", "random_iso") and self.seed is None:
            raise ValueError(
                "random selectors require an explicit seed — an unseeded "
                "matched-random baseline is not reproducible and proposal 4.8 "
                "results computed against it are not reportable"
            )

    def key(self) -> str:
        import hashlib, json
        blob = json.dumps(asdict(self), sort_keys=True, default=str)
        return hashlib.sha256(blob.encode()).hexdigest()[:16]


@dataclass
class AblationResult:
    logits: torch.Tensor              # [n_positions, vocab] ablated next-token logits
    clean_logits: torch.Tensor        # [n_positions, vocab] unablated
    effective_rank: dict[int, torch.Tensor] = field(default_factory=dict)
    spec: AblationSpec | None = None
    ids: torch.Tensor | None = None       # [1, seq_len]; needed to score against
                                          # true next tokens on a corpus (intact side)


def prepare_lens(lens: JacobianLens, device) -> JacobianLens:
    """Move the Jacobians onto the compute device. **Does not touch dtype.**

    ``JacobianLens.__init__`` does ``J.float()`` on every Jacobian, so the class
    holds float32 regardless of what was on disk (``save`` writes fp16 purely
    for compactness). ``apply()`` correspondingly casts residuals with
    ``.float()`` before ``transport``. The library's internal contract is
    float32 throughout, and ``HFLensModel.unembed`` casts to the head's dtype
    itself, so nothing downstream needs the model's dtype here.

    An earlier version of this function cast the Jacobians to the model's dtype.
    That broke the contract and produced
    ``RuntimeError: expected mat1 and mat2 to have the same dtype`` inside
    ``transport``. Callers passing activations straight from
    ``ActivationRecorder`` (which are in the *model's* dtype, not float32) must
    cast those to float — see :func:`build_cache`.

    Only the device move is needed: without it ``transport`` copies a
    ``[d_model, d_model]`` matrix host-to-device on every call.
    """
    lens.jacobians = {k: v.to(device=device) for k, v in lens.jacobians.items()}
    return lens


@dataclass
class PromptCache:
    """Per-prompt work that every condition would otherwise repeat.

    The clean forward pass and the lens readout are identical across all
    conditions for a given prompt — only the direction *selection* differs. A
    37-condition sweep without this recomputes both 37 times.

    What is cached is deliberately small: the ranked token ids per layer, not
    the lens logits themselves. Lens logits are ``[n_positions, vocab]``, which
    at a 150k vocabulary is megabytes per layer per prompt; the ranked ids are
    ``[n_positions, k_max]``. Any ``k <= k_max`` is then a slice.

    ``k_max`` must be at least ``2 * max(k)`` in the sweep, because the
    ``next_k`` selector reads ranks ``k..2k``.
    """

    ids: torch.Tensor
    n_pos: int
    clean_logits: torch.Tensor
    excluded_ids: torch.Tensor | None          # [n_pos, n_exclude]
    ranked_ids: dict[int, torch.Tensor]        # layer -> [n_pos, k_max]
    k_max: int


@torch.no_grad()
def build_cache(
    model: Any, lens: JacobianLens, prompt: str, layers: Sequence[int],
    *, k_max: int, exclude_clean_top: int = 10, max_seq_len: int = 512,
) -> PromptCache:
    """Run the clean pass once and rank directions once, for reuse."""
    ids = model.encode(prompt, max_length=max_seq_len)
    final = model.n_layers - 1
    record_at = sorted({*layers, final})

    with ActivationRecorder(model.layers, record_at) as rec:
        model.forward(ids)
        acts = {i: rec.activations[i][0].detach() for i in record_at}
    clean_logits = model.unembed(acts[final])

    excluded = (clean_top_mask(clean_logits, exclude_clean_top)
                if exclude_clean_top > 0 else None)
    excluded_ids = (clean_logits.topk(exclude_clean_top, dim=-1).indices
                    if exclude_clean_top > 0 else None)

    ranked = {}
    for layer in layers:
        # .float() to match the lens's float32 Jacobians, as apply() does.
        lens_logits = model.unembed(lens.transport(acts[layer].float(), layer))
        ranked[layer] = select_by_rank(lens_logits, k_max, excluded=excluded)

    return PromptCache(ids, ids.shape[1], clean_logits, excluded_ids, ranked, k_max)


class _Ablator:
    """Forward hooks that project out precomputed per-position direction sets."""

    def __init__(
        self,
        blocks: Sequence[torch.nn.Module],
        bases: dict[int, Basis],
        position_mask: torch.Tensor | None,
        mode: str,
    ) -> None:
        self._blocks, self._bases = blocks, bases
        self._position_mask, self._mode = position_mask, mode
        self._handles: list[Any] = []

    def _hook(self, index: int):
        basis = self._bases[index]

        def fn(module, inputs, output):
            is_tuple = not torch.is_tensor(output)
            tensor = output[0] if is_tuple else output
            # tensor: [batch, seq, d_model]; harness runs batch=1.
            h = tensor[0]
            new = project_out(h, basis, mode=self._mode)
            if self._position_mask is not None:
                new = torch.where(self._position_mask[:, None], new, h)
            tensor = torch.cat([new[None], tensor[1:]], dim=0)
            return (tensor, *output[1:]) if is_tuple else tensor

        return fn

    def __enter__(self):
        try:
            for i in self._bases:
                self._handles.append(self._blocks[i].register_forward_hook(self._hook(i)))
        except Exception:
            self.__exit__()
            raise
        return self

    def __exit__(self, *exc) -> None:
        for h in self._handles:
            h.remove()
        self._handles = []


@torch.no_grad()
def run_ablation(
    model: Any,
    lens: JacobianLens,
    unembed_weight: torch.Tensor,
    prompt: str,
    spec: AblationSpec,
    *,
    max_seq_len: int = 512,
    cache: PromptCache | None = None,
) -> AblationResult:
    """Run one ablation condition end to end.

    Args:
        model: Anything satisfying ``jlens.protocol.LensModel``.
        lens: Fitted lens. ``spec.layers`` must be a subset of its source layers
            for lens-based selectors.
        unembed_weight: ``W_U``, ``[vocab, d_model]`` — usually
            ``model.lm_head.weight``. Passed explicitly because the LensModel
            protocol exposes ``unembed()`` (norm + head) but not ``W_U`` itself.
    """
    final = model.n_layers - 1

    # --- Pass 1: clean (skipped entirely when a cache is supplied) ---
    if cache is not None:
        if spec.k * (2 if spec.selector == "next_k" else 1) > cache.k_max:
            raise ValueError(
                f"cache holds k_max={cache.k_max} ranked directions but this "
                f"condition needs {spec.k * (2 if spec.selector == 'next_k' else 1)}. "
                "Rebuild the cache with a larger k_max."
            )
        ids, n_pos = cache.ids, cache.n_pos
        clean_logits, acts = cache.clean_logits, None
    else:
        ids = model.encode(prompt, max_length=max_seq_len)
        n_pos = ids.shape[1]
        with ActivationRecorder(model.layers, sorted({*spec.layers, final})) as rec:
            model.forward(ids)
            acts = {i: rec.activations[i][0].detach() for i in record_at_or(spec, final)}
        clean_logits = model.unembed(acts[final])

    if spec.selector == "none":
        return AblationResult(clean_logits, clean_logits, spec=spec, ids=ids)

    excluded = None
    if spec.exclude_clean_top > 0:
        excluded = (
            _mask_from_ids(cache.excluded_ids, clean_logits.shape)
            if cache is not None
            else clean_top_mask(clean_logits, spec.exclude_clean_top)
        )

    # --- Direction selection, per band layer ---
    gen = torch.Generator(device="cpu")
    if spec.seed is not None:
        gen.manual_seed(spec.seed)
    bases: dict[int, Basis] = {}
    ref = clean_logits
    for layer in spec.layers:
        h = acts[layer] if acts is not None else ref
        if spec.selector == "random_iso":
            vecs = random_isotropic(
                n_pos, spec.k, model.d_model, gen, device=h.device, dtype=h.dtype
            )
        else:
            ranked = cache.ranked_ids[layer] if cache is not None else None
            if ranked is None:
                lens_logits = model.unembed(lens.transport(h.float(), layer))
            if spec.selector == "topk":
                tok = (ranked[:, : spec.k] if ranked is not None
                       else select_by_rank(lens_logits, spec.k, excluded=excluded))
            elif spec.selector == "next_k":
                tok = (ranked[:, spec.k : 2 * spec.k] if ranked is not None
                       else select_by_rank(lens_logits, spec.k,
                                           rank_offset=spec.k, excluded=excluded))
            elif spec.selector == "random_lens":
                tok = random_lens_tokens(
                    n_pos, spec.k, clean_logits.shape[-1], gen,
                    excluded=excluded, device=clean_logits.device,
                )
            else:
                raise ValueError(f"unknown selector {spec.selector!r}")
            vecs = lens_vectors(unembed_weight, lens.jacobians[layer].to(h.device), tok)
        bases[layer] = orthonormalise(vecs)

    pos_mask = None
    if spec.positions is not None:
        pos_mask = torch.zeros(n_pos, dtype=torch.bool, device=ids.device)
        pos_mask[list(spec.positions)] = True

    # --- Pass 2: ablated ---
    with _Ablator(model.layers, bases, pos_mask, spec.mode):
        with ActivationRecorder(model.layers, [final]) as rec2:
            model.forward(ids)
            ablated_final = rec2.activations[final][0].detach()

    return AblationResult(
        logits=model.unembed(ablated_final),
        clean_logits=clean_logits,
        effective_rank={l: b.rank for l, b in bases.items()},
        spec=spec,
        ids=ids,
    )


def greedy_match(result: AblationResult, answer_id: int, position: int = -1) -> dict:
    """Score one prompt: did the greedy next token match, clean and ablated?

    The Control A metric per DECISION_control_A §4.4 — greedy next-token
    accuracy against probe-swap.json's ``answer`` field.
    """
    return {
        "clean_correct": int(result.clean_logits[position].argmax()) == answer_id,
        "ablated_correct": int(result.logits[position].argmax()) == answer_id,
    }

## Cell 4 — Write `band/readout.py`

In [ ]:
%%writefile band/readout.py
"""Stage B2 — readout verification.

Control A ablates the top-k J-lens directions and expects two-hop reasoning to
collapse. That expectation rests on an assumption nothing has yet checked:

    that the top-k directions actually CONTAIN the unspoken intermediate.

For "the language spoken in the country where the Amazon River ends", the paper's
claim is that `Brazil` sits in the readout before the model says `Portuguese`.
If it does not — if the readout surfaces something else — then ablation removes
*something*, but not the thing the theory says matters, and a null becomes
uninterpretable in exactly the way proposal §4.5 warns about ("readout too weak"
vs "phenomenon absent").

`probe-swap.json` carries an `intermediate` field for all 90 prompts, and **none
of the 90 intermediates appears anywhere in its own prompt**, so this is a clean
test of unspoken content rather than an echo of the input.

Two measures, both from the paper:

  rank      position of the intermediate in the lens readout. Low = surfaced.
  loading   cosine similarity between the residual stream and the intermediate's
            lens vector. The paper defines a concept's *workspace loading* this
            way and finds it predicts whether interventions on that concept
            work — number words load poorly and intervene poorly. If your
            intermediates load poorly, that diagnoses a null in advance rather
            than after the fact.

**Foil control.** For each prompt the same measures are computed for a randomly
chosen *other* prompt's intermediate. Without it, a good rank could simply mean
the lens favours common words. The true-vs-foil gap is the actual evidence.

NOTE ON NAMING: this is Phase 0 instrument verification on a language model. It
is not Phase 2's R-space identification, which will live in `src/readout/` and
is a different thing entirely.
"""

from __future__ import annotations

from dataclasses import dataclass, asdict
from typing import Any, Sequence

import torch

from jlens.hooks import ActivationRecorder

from ablation.directions import lens_vectors


@dataclass
class ReadoutRecord:
    """Readout measurements for one prompt at one layer."""

    name: str
    layer: int
    target_token: str
    n_target_tokens: int
    best_rank: int            # best (lowest) rank across positions; 0 = top-1
    best_position: int
    max_loading: float        # cosine sim, best across positions
    mean_loading: float
    is_foil: bool


@torch.no_grad()
def measure_readout(
    model: Any, lens: Any, unembed_weight: torch.Tensor,
    prompt: str, target: str, layers: Sequence[int],
    *, name: str = "", is_foil: bool = False,
    skip_first: int = 4, max_seq_len: int = 128,
) -> list[ReadoutRecord]:
    """Rank and workspace loading of ``target`` in the readout at each layer.

    Args:
        target: the concept word. A leading space is added before tokenising,
            matching how it would appear mid-text.
    """
    tok = model.tokenizer
    tgt = " " + target.strip()
    # Real HF tokenizers take add_special_tokens; the LensModel protocol does not
    # require it (tests/tiny.py's toy tokenizer has no such kwarg), so fall back.
    # A leading BOS from the fallback path is dropped below.
    try:
        ids_t = tok(tgt, add_special_tokens=False).input_ids
    except TypeError:
        ids_t = tok(tgt).input_ids
        if torch.is_tensor(ids_t):
            ids_t = ids_t[0].tolist()
        bos = getattr(tok, "bos_token_id", None)
        if bos is not None and ids_t and ids_t[0] == bos:
            ids_t = ids_t[1:]
    if torch.is_tensor(ids_t):
        ids_t = ids_t[0].tolist() if ids_t.dim() > 1 else ids_t.tolist()
    if not ids_t:
        raise ValueError(f"target {target!r} tokenised to nothing")
    # The J-lens is vocabulary-indexed and only represents single tokens
    # (paper §A.9). For a multi-token target only the first token is
    # representable; n_target_tokens is recorded so those can be split out.
    tid = torch.tensor([ids_t[0]])

    ids = model.encode(prompt, max_length=max_seq_len)
    final = model.n_layers - 1
    with ActivationRecorder(model.layers, sorted({*layers, final})) as rec:
        model.forward(ids)
        acts = {i: rec.activations[i][0].detach() for i in sorted({*layers, final})}

    out = []
    for layer in layers:
        h = acts[layer][skip_first:].float()
        if h.shape[0] == 0:
            continue
        logits = model.unembed(lens.transport(h, layer)).float()

        # rank = how many tokens score strictly higher than the target
        tgt_score = logits[:, tid[0]]
        ranks = (logits > tgt_score[:, None]).sum(-1)
        best_pos = int(ranks.argmin())
        best_rank = int(ranks[best_pos])

        # workspace loading: cosine(residual, the target's lens vector)
        v = lens_vectors(unembed_weight, lens.jacobians[layer].to(h.device),
                         tid.to(h.device))[0].float()
        v = v / v.norm().clamp_min(1e-12)
        cos = (h / h.norm(dim=-1, keepdim=True).clamp_min(1e-12)) @ v

        out.append(ReadoutRecord(
            name=name, layer=layer, target_token=target,
            n_target_tokens=len(ids_t), best_rank=best_rank,
            best_position=best_pos + skip_first,
            max_loading=float(cos.max()), mean_loading=float(cos.mean()),
            is_foil=is_foil,
        ))
    return out


def summarise(records: Sequence[ReadoutRecord], layers: Sequence[int],
              ks: Sequence[int] = (1, 5, 10, 25, 100)) -> dict[str, Any]:
    """Per-layer readout accuracy and loading, true vs foil."""
    out: dict[str, Any] = {"layers": list(layers), "k_values": list(ks), "per_layer": {}}
    for layer in layers:
        row: dict[str, Any] = {}
        for tag, foil in (("true", False), ("foil", True)):
            rs = [r for r in records if r.layer == layer and r.is_foil == foil]
            if not rs:
                continue
            row[tag] = {
                "n": len(rs),
                "median_rank": sorted(r.best_rank for r in rs)[len(rs) // 2],
                "mean_max_loading": sum(r.max_loading for r in rs) / len(rs),
                **{f"top{k}": sum(r.best_rank < k for r in rs) / len(rs) for k in ks},
            }
        if "true" in row and "foil" in row:
            row["top10_gap"] = row["true"]["top10"] - row["foil"]["top10"]
            row["loading_gap"] = row["true"]["mean_max_loading"] - row["foil"]["mean_max_loading"]
        out["per_layer"][layer] = row
    return out


def verdict(summary: dict[str, Any], band: Sequence[int],
            *, min_top10: float = 0.30, min_gap: float = 0.15) -> dict[str, Any]:
    """Does the readout surface intermediates inside the band?

    Thresholds are judgment calls, exposed so the values used are recorded. The
    paper gives no numeric criterion for "the readout works".

    Args:
        min_top10: fraction of prompts whose intermediate must reach the top 10.
        min_gap: how far true must exceed foil on that fraction. This is the
            one that matters — a high top-10 rate with no gap over foils means
            the lens favours common words, not that it surfaced the concept.
    """
    inband = [summary["per_layer"][l] for l in band if l in summary["per_layer"]]
    inband = [r for r in inband if "true" in r and "foil" in r]
    if not inband:
        return {"verdict": "NO DATA"}

    best = max(inband, key=lambda r: r["top10_gap"])
    peak_top10 = max(r["true"]["top10"] for r in inband)
    peak_gap = best["top10_gap"]

    d = {"peak_true_top10": peak_top10, "peak_gap_over_foil": peak_gap,
         "min_top10": min_top10, "min_gap": min_gap}
    if peak_top10 >= min_top10 and peak_gap >= min_gap:
        d["verdict"] = "READOUT SURFACES INTERMEDIATES"
        d["reading"] = (
            f"At its best band layer the intermediate reaches the top 10 for "
            f"{peak_top10:.0%} of prompts, {peak_gap:+.0%} above matched foils. "
            "The premise Control A rests on is supported: the top-k directions "
            "do contain the unspoken intermediate. This is itself a replication "
            "of one of the paper's core claims on an open model."
        )
    elif peak_gap < min_gap:
        d["verdict"] = "NO SIGNAL OVER FOILS"
        d["reading"] = (
            f"True intermediates rank no better than foils (gap {peak_gap:+.0%}). "
            "Whatever the readout is surfacing, it is not prompt-specific "
            "content. Ablating the top-k would remove *something*, but not the "
            "intermediate, and a Control A null could not distinguish 'no "
            "workspace' from 'readout too weak' (proposal §4.5). Report this "
            "and treat any Control A result as bounded by it."
        )
    else:
        d["verdict"] = "WEAK — signal present but below threshold"
        d["reading"] = (
            f"Intermediates beat foils by {peak_gap:+.0%} but reach the top 10 "
            f"for only {peak_top10:.0%} of prompts. The readout carries real "
            "content and is weaker than the paper's. Control A remains "
            "interpretable, with its power bounded by this."
        )
    return d

## Cell 5 — Write `verify_readout.py`

In [ ]:
%%writefile verify_readout.py
"""Stage B2 — does the lens surface the unspoken intermediate?

Run BEFORE Control A. Control A ablates the top-k directions and expects two-hop
reasoning to collapse; that only follows if those directions contain the
intermediate. This checks the premise.

Cheap: one forward pass per prompt, a few minutes, ~1 unit.

Usage:
    python verify_readout.py --model Qwen/Qwen3-8B \
        --lens-file qwen3-8b/jlens/Salesforce-wikitext/Qwen3-8B_jacobian_lens.pt \
        --data jacobian-lens/data/experiments/probe-swap.json \
        --out results/raw/readout_qwen3-8b/
"""
from __future__ import annotations
import argparse, json, random, time
from dataclasses import asdict
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import jlens
from jlens.lens import JacobianLens
from ablation.harness import prepare_lens
from band.readout import measure_readout, summarise, verdict

LENS_REPO = "neuronpedia/jacobian-lens"


@torch.no_grad()
def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--lens-file", required=True)
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--dtype", default="bfloat16")
    ap.add_argument("--band-start", type=int, default=20)   # prereg §2
    ap.add_argument("--band-end", type=int, default=31)
    ap.add_argument("--scan-from", type=int, default=0,
                    help="also measure outside the band, to see where content peaks")
    ap.add_argument("--scan-step", type=int, default=2)
    ap.add_argument("--skip-first", type=int, default=4)
    ap.add_argument("--max-seq-len", type=int, default=128)
    ap.add_argument("--seed", type=int, default=20260728)
    args = ap.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    hf = AutoModelForCausalLM.from_pretrained(args.model, dtype=getattr(torch, args.dtype),
                                              device_map=device)
    lm = jlens.from_hf(hf, AutoTokenizer.from_pretrained(args.model))
    lens = prepare_lens(JacobianLens.from_pretrained(LENS_REPO, filename=args.lens_file), device)
    wu = lm._lm_head.weight.detach()

    items = json.load(open(args.data))["items"]
    band = list(range(args.band_start, args.band_end + 1))
    scan = sorted(set(range(args.scan_from, max(lens.source_layers) + 1, args.scan_step)) | set(band))
    print(f"{len(items)} prompts; measuring {len(scan)} layers "
          f"(band {band[0]}..{band[-1]} plus a scan)\n")

    rng = random.Random(args.seed)
    records = []
    t0 = time.perf_counter()
    for i, it in enumerate(items, 1):
        # foil: another prompt's intermediate. Controls for the lens simply
        # favouring common words.
        foil = rng.choice([x for x in items if x["intermediate"] != it["intermediate"]])
        p = it["prompt"].rstrip()
        records += measure_readout(lm, lens, wu, p, it["intermediate"], scan,
                                   name=it["name"], skip_first=args.skip_first,
                                   max_seq_len=args.max_seq_len)
        records += measure_readout(lm, lens, wu, p, foil["intermediate"], scan,
                                   name=it["name"], is_foil=True,
                                   skip_first=args.skip_first, max_seq_len=args.max_seq_len)
        if i % 15 == 0:
            print(f"  [{i}/{len(items)}] {time.perf_counter()-t0:.0f}s")

    summ = summarise(records, scan)
    v = verdict(summ, band)

    print(f"\n{'layer':>6} {'true top10':>11} {'foil top10':>11} {'gap':>8} "
          f"{'med rank':>9} {'loading':>9} {'foil load':>10}")
    for l in scan:
        r = summ["per_layer"].get(l, {})
        if "true" not in r or "foil" not in r:
            continue
        mark = " *" if args.band_start <= l <= args.band_end else "  "
        print(f"{l:>4}{mark} {r['true']['top10']:>11.1%} {r['foil']['top10']:>11.1%} "
              f"{r['top10_gap']:>+8.1%} {r['true']['median_rank']:>9} "
              f"{r['true']['mean_max_loading']:>9.3f} {r['foil']['mean_max_loading']:>10.3f}")
    print("  (* = inside the pre-registered band)")

    print(f"\nVERDICT: {v['verdict']}")
    print(f"  {v.get('reading','')}")

    out = Path(args.out); out.mkdir(parents=True, exist_ok=True)
    (out / "readout_records.json").write_text(
        json.dumps([asdict(r) for r in records], indent=2))
    (out / "readout_summary.json").write_text(
        json.dumps({"summary": summ, "verdict": v, "band": band,
                    "config": vars(args)}, indent=2, default=str))
    print(f"\nwrote {out}/readout_summary.json")


if __name__ == "__main__":
    main()

## Cell 6 — Run it

Scans every other layer plus the full pre-registered band (L20–31, marked `*`).

In [ ]:
!python verify_readout.py \
    --model Qwen/Qwen3-8B \
    --lens-file qwen3-8b/jlens/Salesforce-wikitext/Qwen3-8B_jacobian_lens.pt \
    --data jacobian-lens/data/experiments/probe-swap.json \
    --out results/raw/readout_qwen3-8b/ \
    --band-start 20 --band-end 31 \
    --scan-from 0 --scan-step 2

## Cell 7 — Plot

Top: how often the intermediate reaches the top 10, true vs foil. **The gap between the lines is the result.** Bottom: workspace loading.

In [ ]:
import json, matplotlib.pyplot as plt

D = json.load(open("results/raw/readout_qwen3-8b/readout_summary.json"))
P, band = D["summary"]["per_layer"], D["band"]
ls = sorted(int(k) for k in P if "true" in P[k] and "foil" in P[k])

fig, ax = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
ax[0].plot(ls, [P[str(l)]["true"]["top10"] for l in ls], "o-", label="true intermediate")
ax[0].plot(ls, [P[str(l)]["foil"]["top10"] for l in ls], "s--", c="tab:gray", label="foil")
ax[0].set_ylabel("fraction reaching top 10"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[0].set_title("Does the readout surface the unspoken intermediate?", loc="left")

ax[1].plot(ls, [P[str(l)]["true"]["mean_max_loading"] for l in ls], "o-", label="true")
ax[1].plot(ls, [P[str(l)]["foil"]["mean_max_loading"] for l in ls], "s--",
           c="tab:gray", label="foil")
ax[1].set_ylabel("workspace loading (cosine)"); ax[1].set_xlabel("layer")
ax[1].legend(); ax[1].grid(alpha=.3)

for a in ax:
    a.axvspan(band[0], band[-1], alpha=.12, color="tab:green")
fig.suptitle(f"{D['config']['model']} — green = pre-registered band "
             f"L{band[0]}-{band[-1]}", fontsize=10)
fig.tight_layout()
plt.savefig("results/raw/readout_qwen3-8b/readout_curves.png", dpi=130)
plt.show()
print("VERDICT:", D["verdict"]["verdict"])

## Cell 8 — Download

In [ ]:
from google.colab import files
files.download("results/raw/readout_qwen3-8b/readout_summary.json")
files.download("results/raw/readout_qwen3-8b/readout_curves.png")

---
## Report back

Paste the cell 6 table and verdict, and attach the plot.

### What each outcome would mean for Control A

**Solid gap over foils inside the band** — the premise holds. Run Control A as pre-registered. This is also a positive result in its own right: a replication of one of the paper's core claims on an open model, independent of whether ablation works.

**No gap over foils** — ablating the top-k would remove *something*, but not the intermediate. Control A becomes much weaker, and a null could not distinguish "no workspace" from "readout too weak" (proposal §4.5). Worth pausing to reconsider the band or the lens recipe before spending the hour.

**Content peaks outside the band** — if the gap is largest at, say, L12–18 rather than L20–31, the band derived from aggregate statistics may not be where task-relevant content lives. That would be an argument for adjusting the band, logged as an amendment with this as the stated evidence.